## tl;dr

Matched results do not show a uniform monotonic wind-strength effect. High/Medium changes are large and mixed in sign; Low/Medium is usually closer but has larger local exceptions. No eligible within-condition replicate cells remain, so random variation and wind-related effects cannot be separated.


## Context & Methods

Same wind direction, formation, spacing, drone, and battery; compare Lv1 versus Lv2. Exclude the initial SOC plateau only.

### Key Assumptions

Recorded levels are treatment labels, not measured local m/s. Within-flight drone observations are correlated. The primary cohort requires complete curves; the sensitivity cohort needs only complete pairs of stages.


## Data

Data and code are local to the drone_experiment project. Sources are read-only; no model or flight-control code is changed.


In [1]:
from pathlib import Path
import sys,json,sqlite3
import pandas as pd
import numpy as np
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'battery_normalization.py').exists())
sys.path.insert(0,str(ROOT/'output_py'))
from diagnose_wind_strength_stage_ratios import OUT,PRIOR,SQL,describe
from audit_wind_tunnel_stage_ratios import sha
full=pd.read_csv(PRIOR/'paired_ratios.csv')
with sqlite3.connect(':memory:') as conn:
    full.to_sql('full_curves',conn,index=False)
    matched=pd.read_sql_query(SQL,conn)
print(matched[['wind_direction','formation','battery_id','high_change_pct','low_change_pct']].round(3).to_string(index=False))


wind_direction formation battery_id  high_change_pct  low_change_pct
          head     front        B11           18.912           2.412
          side   diamond        B10          -29.569           3.731
          side   diamond        B11          -27.209           2.823
          side     front        B11            8.747           6.043
          tail   diamond        B11          -37.913           2.697
          tail     front        B10           -2.139          -4.091
          tail     front        B11           38.807          -4.718


### Check provenance and pairing


In [2]:
audit=json.loads((OUT/'summary.json').read_text())
for source,expected in audit['input_hashes'].items():
    assert sha(ROOT/source)==expected
saved=pd.read_csv(OUT/'matched_full_curves.csv')
for stage in ['high','low']:
    assert np.allclose(matched[stage+'_change_pct'],saved[stage+'_change_pct'])
assert not full.duplicated(['wind_direction','formation','spacing_cm','battery_id','drone_id','wind_level']).any()
print('Source hashes, independent SQL matches and primary one-to-one joins: passed')


Source hashes, independent SQL matches and primary one-to-one joins: passed


## Results


In [3]:
for stage in ['high','low']:
    print('Primary',stage,describe(matched[stage+'_change_pct']))
extended=pd.read_csv(OUT/'expanded_matched_pairs.csv')
for stage,g in extended.groupby('stage'):
    print('Extended',stage,describe(g.change_pct))
repeats=pd.read_csv(OUT/'repeated_cells.csv')
print('Eligible replicate cells:',len(repeats))
common=pd.read_csv(OUT/'common_prelanding_low.csv')
print(common[common.included][['wind_direction','formation','battery_id','end_soc','change_pct']].to_string(index=False))


Primary high {'n': 7, 'median': -2.1389486181879303, 'median_abs': 27.208660425395514, 'mean': -4.337770694138997, 'mean_abs': 23.328079094825995, 'minimum': -37.91343265017312, 'maximum': 38.8068062130077, 'increased': 3, 'decreased': 4, 'within10': 2}
Primary low {'n': 7, 'median': 2.6966101260986974, 'median_abs': 3.7306775202282205, 'mean': 1.2707658174913816, 'mean_abs': 3.787705544780126, 'minimum': -4.717800412056549, 'maximum': 6.042613569010324, 'increased': 5, 'decreased': 2, 'within10': 7}
Extended high {'n': 10, 'median': 8.726233119925, 'median_abs': 28.389046496600002, 'mean': 14.699957377336, 'mean_abs': 34.06605222967401, 'minimum': -37.9134326507, 'maximum': 97.5807370082, 'increased': 6, 'decreased': 4, 'within10': 3}
Extended low {'n': 26, 'median': 0.03174389074549999, 'median_abs': 4.4046445221349995, 'mean': 1.405275005437731, 'mean_abs': 7.808152731550807, 'minimum': -22.7450782832, 'maximum': 25.5372687907, 'increased': 13, 'decreased': 13, 'within10': 18}
Eligi

## Takeaways

No universal wind-level multiplier is supported. Configuration-specific differences are descriptive hypotheses, not demonstrated causation. The missing within-condition repeats prevent an empirical estimate of repeatability noise. Expanded Low pairs include landing-related environment changes; the matched pre-landing check is limited to three pairs.

Execution: all code cells above ran sequentially in Python. Jupyter-kernel execution was not available (nbformat/nbclient/ipykernel not installed). To validate in a Jupyter environment, run `python -m jupyter nbconvert --execute --to notebook --inplace wind_strength_audit.ipynb` from this directory.
